In [19]:
import os, sys

nb_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Using project root:', project_root)
print('First sys.path entry:', sys.path[0])

Using project root: /Users/rithvik/Documents/hnrs/Decoder
First sys.path entry: /Users/rithvik/Documents/hnrs/Decoder


In [20]:
import numpy as np
from scipy.sparse import csr_matrix, eye, hstack, save_npz, load_npz
import scipy.io

In [21]:
from utils.LDPC_encode import LDPCEncode
from utils.awgn_channel import AWGNChannel
from utils.find_ber import findBER

In [22]:
from ldpc.bp_decoder import BpDecoder

In [23]:
H_mat_dat = scipy.io.loadmat('H.mat')
H = csr_matrix(H_mat_dat['H'])

In [24]:
n = 486 # length of message
n_frames = 10000
max_iter = 30

message = np.random.randint(0, 2, (n_frames, n))
print("Message shape:", message.shape)

Message shape: (10000, 486)


In [25]:
encoded_codeword = LDPCEncode(message)
print("Encoded codeword shape:", encoded_codeword.shape) 

tx_codeword = 1 - 2 * encoded_codeword 

Encoded codeword shape: (10000, 648)


In [26]:
m, _ = H.shape
arr = np.arange(m)
schedule = arr.reshape(6, -1)

print(m)


162


In [27]:
snrs = [-3, -2, -1, 0, 1, 2, 3, 4]
bers = []

for snr in snrs:
    rx_llrs = AWGNChannel(tx_codeword, snr_db=snr)
    decoded_codewords = []

    for i in range(n_frames):
        llr = rx_llrs[i, :]

        decoder.reset()
        decoder.initialise_log_domain_bp(llr)
        for iter in range(max_iter):
            x = np.random.randint(0, 6)
            cluster = schedule[x]
            llr = decoder.decode_cluster(cluster)
        
        decoded_codeword = (llr < 0).astype(int)
        decoded_codewords.append(decoded_codeword)
        
    decoded_codewords = np.array(decoded_codewords)
    decoded_message = decoded_codewords[:, :n]
    ber = findBER(message, decoded_message)
    bers.append(ber)
    print(f"BER at SNR {snr} dB: {ber}")

BER at SNR -3 dB: 0.1584738683127572
BER at SNR -2 dB: 0.12991481481481482
BER at SNR -1 dB: 0.10113106995884774
BER at SNR 0 dB: 0.0700417695473251
BER at SNR 1 dB: 0.03028230452674897
BER at SNR 2 dB: 0.0021613168724279835
BER at SNR 3 dB: 1.7695473251028805e-05
BER at SNR 4 dB: 6.17283950617284e-07
